In [ ]:
import xarray as xr, netCDF4 as nc, numpy as np, pandas as pd, os
from pathlib import Path

import dask
import dask.array as da
from dask.distributed import LocalCluster, Client
from datetime import datetime

In [ ]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
workingDir = Path().absolute()
print(f"{workingDir}")

In [ ]:
client = Client()
client

In [ ]:
tas_path = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/tas/latest/"
write_path = f'{workingDir}/data/preprocess/'

In [ ]:
sdate, edate ='19790101', '20001231'

In [ ]:
fdates = [m.strftime('%Y%m') for m in pd.date_range(sdate, edate, freq='ME')]
fnames = [s for s in os.listdir(tas_path) if any(f in s for f in fdates)]
fpaths = sorted([tas_path + f for f in fnames])

tas_ds = xr.open_mfdataset(fpaths, concat_dim ='time', combine='nested', 
                           parallel=True, data_vars='minimal',coords='minimal', 
                           drop_variables = "time_bnds",chunks="auto")
datestr = f"s{sdate}_e{edate}"

t95 = tas_ds.reduce(np.nanpercentile, q=95, dim="time")
t95 = t95.rename(name_dict={'tas':'PRCTILE95'})

In [ ]:
encoding = {"PRCTILE95":{"zlib": True, "complevel": 4, "shuffle": True}}

write_task = t95.to_netcdf(f'{write_path}t95_baseline.nc',
                                        encoding=encoding,
                                        compute=False,
                                        engine="netcdf4")
dask.compute(write_task)